# OEM Fleet Recall Propagation - Snowsight Notebook

Snowsight variant of `cars_demo.ipynb`. The five acts walk through `CARS_DEMO.FLEET` (325 vehicles, 762 service events, 254 recall assignments, 5 active campaigns) with one RAI reasoner family per act on the same `cars` PyRel ontology.

Prerequisites:
1. RelationalAI native app installed; account-level role `RAI_DEVELOPER` granted to the running user.
2. The `cars` model uses named engines `cars_logic_l` (HIGHMEM_X64_L) and `cars_prescriptive_m` (HIGHMEM_X64_M). They auto-resume on first query (~3-5 min cold).
3. `CHANGE_TRACKING = TRUE` on every `CARS_DEMO.FLEET.*` table (set during the loader).

In [ ]:
# Install pinned versions known to work together (matches the
# local .venv that runs cars_demo.ipynb). The Snowsight container
# bundles older versions at /opt/venv/snowbook/...; --force-reinstall
# replaces them with these.
import sys, subprocess

out = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--upgrade', '--force-reinstall',
     '--no-warn-script-location',
     'protobuf==5.29.6',
     'lqp==0.5.2',
     'relationalai==1.7.1',
     'plotly', 'networkx', 'pandas'],
    capture_output=True, text=True, check=False,
)
print('=== PIP STDOUT (tail) ===')
print(out.stdout[-3000:])
if out.returncode != 0:
    print('=== PIP STDERR ===')
    print(out.stderr[-2500:])
    raise SystemExit(f'pip failed rc={out.returncode}')

# Wipe any pre-cached module references so re-import resolves to
# the freshly installed code.
for mod in list(sys.modules):
    if mod.startswith(('google.protobuf', 'lqp', 'relationalai', 'v0')):
        del sys.modules[mod]

import google.protobuf, lqp, relationalai
print(f'protobuf {google.protobuf.__version__}')
print(f'lqp at {lqp.__file__}')
print(f'relationalai {relationalai.__version__} at {relationalai.__file__}')

In [ ]:
# Snowsight inserts get_active_session() into scope automatically;
# the cars._build_config() helper picks it up via
# ConfigFromActiveSession when running here. No nest_asyncio needed.
from snowflake.snowpark.context import get_active_session
session = get_active_session()

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Importing the ontology triggers all model.define() statements.
from cars import (
    Supplier, Part, BomNode, Vehicle, Owner, ServiceCentre, Region,
    RecallCampaign, RecallAssignment, OpenRecall, SLABreachedRecall,
    PriorAccident, PriorityVehicle, in_bom, model,
)
from demo_queries import (
    q1_recall_sla_audit, q1b_breached_by_centre,
    q2_continental_cascade, q2_regional_rollup, q2_top_centres,
    q3_urgency_top20,
    q4_assign_recall_jobs, q5_assign_recall_jobs_priority,
)
print('imports ok')

: 

## Act 1 - Rules: recall SLA compliance audit

> **The question:** 'Show me every Open recall on a VIN whose owner-notification age has breached the campaign's completion window. Break it out by campaign.'

The `SLABreachedRecall` derived concept encodes NHTSA 49 CFR 577 / KBA Rueckruf semantics once: `OpenRecall AND age_days_at_demo > completion_days AND severity_code <= 2`.

In [ ]:
df1 = q1_recall_sla_audit()
df1

In [ ]:
fig = px.bar(
    df1, x='campaign', y='breached_open', color='severity',
    color_continuous_scale=[(0, '#c62828'), (0.5, '#ef6c00'), (1, '#f9a825')],
    range_color=(1, 3),
    title='SLA-breached Open recalls by campaign',
    text='breached_open', hover_data=['campaign_name'], width=1000, height=420,
)
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_title='Campaign', yaxis_title='SLA-breached Open recalls')
fig

## Act 2 - Graph: defect cascade from Continental MK C1

In [ ]:
df2 = q2_continental_cascade()
print(f'affected VINs: {len(df2)}')
df2.head(15)

In [ ]:
df2_rollup = q2_regional_rollup()
df2_centres = q2_top_centres()
fig = make_subplots(rows=1, cols=2, subplot_titles=('Regional rollup', 'Top centres by affected VINs'))
fig.add_trace(go.Bar(x=df2_rollup['rollup'], y=df2_rollup['affected_vins'], text=df2_rollup['affected_vins'], textposition='outside'), row=1, col=1)
fig.add_trace(go.Bar(x=df2_centres['centre'], y=df2_centres['affected_vins'], text=df2_centres['affected_vins'], textposition='outside', marker_color='#e57373'), row=1, col=2)
fig.update_xaxes(tickangle=-30, row=1, col=2)
fig.update_layout(showlegend=False, title_text='Continental MK C1 brake-booster cascade', width=1100, height=460)
fig

## Act 3 - Heuristic: per-VIN urgency ranking (top 20)

In [ ]:
df3 = q3_urgency_top20()
df3

In [ ]:
df3_plot = df3.copy()
df3_plot['label'] = df3_plot['vin'].str[-6:] + ' | ' + df3_plot['model'].str.slice(0, 22) + ' | ' + df3_plot['campaign']
fig = px.bar(
    df3_plot.sort_values('urgency'), y='label', x='urgency', orientation='h',
    color='accident', color_discrete_sequence=px.colors.qualitative.Set2,
    title='Top-20 Open recalls by urgency score',
    text='urgency', hover_data=['plant', 'mileage', 'age_days', 'distance_km'],
    width=1100, height=620,
)
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(yaxis_title='Vehicle', xaxis_title='Urgency score (0 = lowest)')
fig

## Act 4 - Prescriptive: assign jobs to centres x 4 weeks

In [ ]:
df4, si4 = q4_assign_recall_jobs()
print(f'status: {si4.termination_status}    objective: {si4.objective_value:.2f}    solve time: {si4.solve_time_sec:.2f}s    jobs scheduled: {len(df4)}')
df4.head(15)

In [ ]:
df4_pivot = df4.groupby(['centre', 'week']).size().reset_index(name='jobs')
fig = px.bar(
    df4_pivot, x='centre', y='jobs', color='week',
    color_continuous_scale='Viridis',
    title='Recall job assignments by centre and week (Act 4 baseline)',
    barmode='stack', width=1100, height=520,
)
fig.update_layout(xaxis_tickangle=-30, yaxis_title='Jobs scheduled')
fig

## Act 5 - Persistent rule: prior-accident VINs go to week 1 or 2

In [ ]:
df5, si5 = q5_assign_recall_jobs_priority()
print(f'status: {si5.termination_status}    objective: {si5.objective_value:.2f}    solve time: {si5.solve_time_sec:.2f}s    jobs scheduled: {len(df5)}')
df5.head(15)

In [ ]:
wk4 = df4.groupby('week').size().reindex([1, 2, 3, 4], fill_value=0)
wk5 = df5.groupby('week').size().reindex([1, 2, 3, 4], fill_value=0)
fig = make_subplots(rows=1, cols=2, subplot_titles=('Jobs per week', 'Objective: weighted lateness'))
fig.add_trace(go.Bar(name='Act 4', x=wk4.index, y=wk4.values, marker_color='#5e81ac'), row=1, col=1)
fig.add_trace(go.Bar(name='Act 5', x=wk5.index, y=wk5.values, marker_color='#bf616a'), row=1, col=1)
fig.add_trace(go.Bar(name='objective', x=['Act 4', 'Act 5'], y=[si4.objective_value, si5.objective_value], marker_color=['#5e81ac', '#bf616a']), row=1, col=2)
fig.update_xaxes(title_text='Week', row=1, col=1)
fig.update_xaxes(title_text='Solve', row=1, col=2)
fig.update_yaxes(title_text='Jobs scheduled', row=1, col=1)
fig.update_yaxes(title_text='Total weighted lateness', row=1, col=2)
fig.update_layout(barmode='group', title_text='Act 4 vs Act 5 - persistent rule effect', width=1100, height=460)
fig

**Closing.** Five acts, five reasoners, one ontology, one schema. The Act 5 rule (`PriorityVehicle = OpenRecall + PriorAccident`) is now structural in the ontology - it lights up in Act 1's SLA audit, in Act 3's urgency score, and in Act 4 / Act 5 of any future MIP. Institutional knowledge moved from a person to a data model.